In [ ]:
!git clone https://github.com/chiiii3/DS-340W.git

In [ ]:
!unzip DS-340W/drug_review_train.csv.zip
!unzip DS-340W/drug_review_test.csv.zip
!unzip DS-340W/drug_review_validation.csv.zip

Archive:  DS-340W/drug_review_train.csv.zip
replace drug_review_train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import pandas as pd

df_train = pd.read_csv("drug_review_train.csv")
df_test = pd.read_csv("drug_review_test.csv")
df_validation = pd.read_csv("drug_review_validation.csv")

df_train.head()

In [ ]:
df_train['polarity'] = df_train['rating'].apply(lambda x: 0 if x <= 5 else 1)
display(df_train.head())

In [ ]:
df_test['polarity'] = df_test['rating'].apply(lambda x: 0 if x <= 5 else 1)
display(df_test.head())

In [ ]:
df_validation['polarity'] = df_validation['rating'].apply(lambda x: 0 if x <= 5 else 1)
display(df_validation.head())

## Cleaning Text Data

### Subtask:
Cleaning the 'review' column in `df_train`, `df_test`, and `df_validation` by removing HTML tags, special characters, and numbers.



In [ ]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return text
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove special characters and numbers, keep only alphabetic characters and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df_train['review'] = df_train['review'].apply(clean_text)
df_test['review'] = df_test['review'].apply(clean_text)
df_validation['review'] = df_validation['review'].apply(clean_text)

display(df_train.head())

## Tokenize and Lowercase Text

### Subtask:
Tokenize the words in the 'review' column of `df_train`, `df_test`, and `df_validation`, and convert all tokens to lowercase.

In [ ]:
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

print("NLTK punkt tokenizer and punkt_tab are ready.")

In [ ]:
from nltk.tokenize import word_tokenize

def tokenize_and_lowercase(text):
    if not isinstance(text, str):
        return []
    tokens = word_tokenize(text)
    return [word.lower() for word in tokens]

df_train['review'] = df_train['review'].apply(tokenize_and_lowercase)
df_test['review'] = df_test['review'].apply(tokenize_and_lowercase)
df_validation['review'] = df_validation['review'].apply(tokenize_and_lowercase)

display(df_train.head())

## Remove Stopwords

### Subtask:
Removing common English stopwords from the tokenized 'review' column in `df_train`, `df_test`, and `df_validation`.

In [ ]:
import nltk
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

print("NLTK stopwords corpus is ready.")

In [ ]:
from nltk.corpus import stopwords

# Define a set of English stopwords for efficient lookup
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    if not isinstance(tokens, list):
        return []
    return [word for word in tokens if word not in stop_words]

df_train['review'] = df_train['review'].apply(remove_stopwords)
df_test['review'] = df_test['review'].apply(remove_stopwords)
df_validation['review'] = df_validation['review'].apply(remove_stopwords)

display(df_train.head())

## Lemmatize Text

### Subtask:

Lemmatize the words in the 'review' column of `df_train`, `df_test`, and `df_validation`.

In [ ]:
import nltk
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

try:
    nltk.data.find('corpora/omw-1.4')
except LookupError:
    nltk.download('omw-1.4')

print("NLTK wordnet and omw-1.4 corpora are ready.")

In [ ]:
from nltk.stem import WordNetLemmatizer

# Initialize WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatize_words(tokens):
    if not isinstance(tokens, list):
        return []
    return [lemmatizer.lemmatize(word) for word in tokens]

df_train['review'] = df_train['review'].apply(lemmatize_words)
df_test['review'] = df_test['review'].apply(lemmatize_words)
df_validation['review'] = df_validation['review'].apply(lemmatize_words)

display(df_train.head())

## Feature Extraction: TF-IDF Vectorization

### Subtask:
Convert the preprocessed text data in the 'review' column of `df_train`, `df_test`, and `df_validation` into numerical feature vectors using TF-IDF.

In [ ]:
# Extract the target variable (polarity) for training, testing, and validation sets
y_train = df_train['polarity']
y_test = df_test['polarity']
y_validation = df_validation['polarity']

print("Target variables y_train, y_test, and y_validation have been created.")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")
print(f"Shape of y_validation: {y_validation.shape}")


In [ ]:
# Join the list of lemmatized words back into a single string for TF-IDF processing
df_train['review_text'] = df_train['review'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
df_test['review_text'] = df_test['review'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
df_validation['review_text'] = df_validation['review'].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')

print("Joined lemmatized tokens into strings for TF-IDF.")


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting to 5000 features for manageable size

# Fit the vectorizer on the training data and transform it
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train['review_text'])

# Transform the test and validation data using the fitted vectorizer
X_test_tfidf = tfidf_vectorizer.transform(df_test['review_text'])
X_validation_tfidf = tfidf_vectorizer.transform(df_validation['review_text'])

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_test_tfidf: {X_test_tfidf.shape}")
print(f"Shape of X_validation_tfidf: {X_validation_tfidf.shape}")

print("TF-IDF vectorization complete. The features are stored in X_train_tfidf, X_test_tfidf, and X_validation_tfidf.")


## Naive Bayes Model for Sentiment Analysis

### Subtask:
Implement a Naive Bayes model for sentiment analysis using the TF-IDF features. Train the model, make predictions, and evaluate its performance on the test and validation datasets.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Initialize the Naive Bayes classifier (MultinomialNB is suitable for TF-IDF features)
naive_bayes_model = MultinomialNB()

# Train the model
naive_bayes_model.fit(X_train_tfidf, y_train)

print("Naive Bayes model trained successfully.")

In [ ]:
# Make predictions on the test set
y_pred_test = naive_bayes_model.predict(X_test_tfidf)

# Evaluate the model on the test set
accuracy_test = accuracy_score(y_test, y_pred_test)
precision_test = precision_score(y_test, y_pred_test)
recall_test = recall_score(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)

print("--- Naive Bayes Model Performance on Test Set ---")
print(f"Accuracy: {accuracy_test:.4f}")
print(f"Precision: {precision_test:.4f}")
print(f"Recall: {recall_test:.4f}")
print(f"F1-Score: {f1_test:.4f}")

# Display classification report for the test set
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_test))

# Display confusion matrix for the test set
print("\nConfusion Matrix (Test Set):")
print(confusion_matrix(y_test, y_pred_test))

In [ ]:
# Make predictions on the validation set
y_pred_validation = naive_bayes_model.predict(X_validation_tfidf)

# Evaluate the model on the validation set
accuracy_validation = accuracy_score(y_validation, y_pred_validation)
precision_validation = precision_score(y_validation, y_pred_validation)
recall_validation = recall_score(y_validation, y_pred_validation)
f1_validation = f1_score(y_validation, y_pred_validation)

print("--- Naive Bayes Model Performance on Validation Set ---")
print(f"Accuracy: {accuracy_validation:.4f}")
print(f"Precision: {precision_validation:.4f}")
print(f"Recall: {recall_validation:.4f}")
print(f"F1-Score: {f1_validation:.4f}")

# Display classification report for the validation set
print("\nClassification Report (Validation Set):")
print(classification_report(y_validation, y_pred_validation))

# Display confusion matrix for the validation set
print("\nConfusion Matrix (Validation Set):")
print(confusion_matrix(y_validation, y_pred_validation))

## Support Vector Machine (SVM) Model for Sentiment Analysis

### Subtask:
Implement an SVM model for sentiment analysis using the TF-IDF features. Train the model, make predictions, and evaluate its performance on the test and validation datasets.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Initialize the LinearSVC classifier (suitable for large datasets and text classification)
svm_model = LinearSVC(random_state=42, dual=False) # dual=False is recommended for n_samples > n_features

# Train the model
svm_model.fit(X_train_tfidf, y_train)

print("SVM model trained successfully.")

In [ ]:
# Make predictions on the test set
y_pred_test_svm = svm_model.predict(X_test_tfidf)

# Evaluate the SVM model on the test set
accuracy_test_svm = accuracy_score(y_test, y_pred_test_svm)
precision_test_svm = precision_score(y_test, y_pred_test_svm)
recall_test_svm = recall_score(y_test, y_pred_test_svm)
f1_test_svm = f1_score(y_test, y_pred_test_svm)

print("--- SVM Model Performance on Test Set ---")
print(f"Accuracy: {accuracy_test_svm:.4f}")
print(f"Precision: {precision_test_svm:.4f}")
print(f"Recall: {recall_test_svm:.4f}")
print(f"F1-Score: {f1_test_svm:.4f}")

# Display classification report for the test set
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_test_svm))

# Display confusion matrix for the test set
print("\nConfusion Matrix (Test Set):")
print(confusion_matrix(y_test, y_pred_test_svm))

In [ ]:
# Make predictions on the validation set
y_pred_validation_svm = svm_model.predict(X_validation_tfidf)

# Evaluate the SVM model on the validation set
accuracy_validation_svm = accuracy_score(y_validation, y_pred_validation_svm)
precision_validation_svm = precision_score(y_validation, y_pred_validation_svm)
recall_validation_svm = recall_score(y_validation, y_pred_validation_svm)
f1_validation_svm = f1_score(y_validation, y_pred_validation_svm)

print("--- SVM Model Performance on Validation Set ---")
print(f"Accuracy: {accuracy_validation_svm:.4f}")
print(f"Precision: {precision_validation_svm:.4f}")
print(f"Recall: {recall_validation_svm:.4f}")
print(f"F1-Score: {f1_validation_svm:.4f}")

# Display classification report for the validation set
print("\nClassification Report (Validation Set):")
print(classification_report(y_validation, y_pred_validation_svm))

# Display confusion matrix for the validation set
print("\nConfusion Matrix (Validation Set):")
print(confusion_matrix(y_validation, y_pred_validation_svm))

## CNN Model for Sentiment Analysis

### Subtask: Data Preparation for CNN

Prepare the preprocessed text data for input into a Convolutional Neural Network (CNN) model. This involves:
1.  **Tokenization and Sequencing:** Convert the lemmatized word lists into numerical sequences.
2.  **Padding Sequences:** Ensure all sequences have a uniform length.
3.  **Define Embedding Parameters:** Determine vocabulary size and embedding dimension.

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Combine all reviews to fit the tokenizer
all_reviews_for_cnn = [
    review for review_list in df_train['review'] for review in review_list
] + [
    review for review_list in df_test['review'] for review in review_list
] + [
    review for review_list in df_validation['review'] for review in review_list
]

# 1. Fit tokenizer on all reviews to create word-to-index mapping
# Limiting num_words to 5000 (similar to TF-IDF) and adding an OOV token
tokenizer_cnn = Tokenizer(num_words=5000, oov_token="<unk>")
tokenizer_cnn.fit_on_texts(df_train['review'])

# Convert text (lists of words) to sequences of integers
X_train_seq = tokenizer_cnn.texts_to_sequences(df_train['review'])
X_test_seq = tokenizer_cnn.texts_to_sequences(df_test['review'])
X_validation_seq = tokenizer_cnn.texts_to_sequences(df_validation['review'])

print("Reviews converted to numerical sequences.")

In [ ]:
# 2. Pad sequences to ensure uniform length

# Determine max sequence length (e.g., 99th percentile of review lengths in training data)
# This helps to capture most review information without excessively long sequences
review_lengths = [len(s) for s in X_train_seq]
max_len = int(np.quantile(review_lengths, 0.99)) + 1 # Add 1 to be safe

# Pad sequences with zeros (post-padding means zeros are added at the end)
X_train_padded = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=max_len, padding='post')
X_validation_padded = pad_sequences(X_validation_seq, maxlen=max_len, padding='post')

print(f"Max sequence length determined to be: {max_len}")
print(f"Shape of X_train_padded: {X_train_padded.shape}")
print(f"Shape of X_test_padded: {X_test_padded.shape}")
print(f"Shape of X_validation_padded: {X_validation_padded.shape}")

In [ ]:
# 3. Define embedding parameters

vocab_size = 5000
embedding_dim = 100

print(f"Vocabulary size for CNN: {vocab_size}")
print(f"Embedding dimension for CNN: {embedding_dim}")


### Subtask: Build and Train the CNN Model

Now, let's define the architecture of the CNN model, compile it, and train it using the prepared data.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

# Define the CNN model architecture
cnn_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(), # Extracts the most important features
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary classification (positive/negative sentiment)
])

# Compile the model
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

cnn_model.summary()
print("CNN model defined and compiled successfully.")


In [ ]:
# Train the CNN model
epochs = 5 # Number of training epochs
batch_size = 32 # Size of the training batches

history = cnn_model.fit(
    X_train_padded, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_validation_padded, y_validation)
)

print("CNN model training complete.")

### Subtask: Evaluate the CNN Model

Evaluate the performance of the trained CNN model on the test dataset.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on the test set
y_pred_proba_cnn = cnn_model.predict(X_test_padded)
y_pred_cnn = (y_pred_proba_cnn > 0.5).astype(int) # Convert probabilities to binary predictions

# Evaluate the model on the test set
print("--- CNN Model Performance on Test Set ---")
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_cnn))

print("\nConfusion Matrix (Test Set):")
print(confusion_matrix(y_test, y_pred_cnn))


## LSTM Model for Sentiment Analysis

### Subtask:
Implement an LSTM model for sentiment analysis using the prepared sequence data. Train the model, make predictions, and evaluate its performance on the test dataset.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Define the LSTM model architecture
lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    LSTM(128), # LSTM layer with 128 units
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary classification
])

# Compile the model
lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

lstm_model.summary()
print("LSTM model defined and compiled successfully.")

### Subtask: Train the LSTM Model

In [ ]:
# Train the LSTM model
epochs = 5 # Number of training epochs
batch_size = 32 # Size of the training batches

history_lstm = lstm_model.fit(
    X_train_padded, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_validation_padded, y_validation)
)

print("LSTM model training complete.")

### Subtask: Evaluate the LSTM Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on the test set
y_pred_proba_lstm = lstm_model.predict(X_test_padded)
y_pred_lstm = (y_pred_proba_lstm > 0.5).astype(int) # Convert probabilities to binary predictions

# Evaluate the model on the test set
print("--- LSTM Model Performance on Test Set ---")
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_lstm))

print("\nConfusion Matrix (Test Set):")
print(confusion_matrix(y_test, y_pred_lstm))

## CNN-LSTM Model for Sentiment Analysis

### Subtask:
Implement a CNN-LSTM model for sentiment analysis using the prepared sequence data. Train the model, make predictions, and evaluate its performance on the test dataset.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, LSTM, Dense, Dropout

# Define the CNN-LSTM model architecture
cnn_lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=max_len - 5 + 1), # Max pooling over the entire sequence after Conv1D
    LSTM(128), # LSTM layer with 128 units
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Binary classification
])

# Compile the model
cnn_lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

cnn_lstm_model.summary()
print("CNN-LSTM model defined and compiled successfully.")

### Subtask: Train the CNN-LSTM Model

In [ ]:
# Train the CNN-LSTM model
epochs = 5 # Number of training epochs
batch_size = 32 # Size of the training batches

history_cnn_lstm = cnn_lstm_model.fit(
    X_train_padded, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_validation_padded, y_validation)
)

print("CNN-LSTM model training complete.")

### Subtask: Evaluate the CNN-LSTM Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Make predictions on the test set
y_pred_proba_cnn_lstm = cnn_lstm_model.predict(X_test_padded)
y_pred_cnn_lstm = (y_pred_proba_cnn_lstm > 0.5).astype(int) # Convert probabilities to binary predictions

# Evaluate the model on the test set
print("-- CNN-LSTM Model Performance on Test Set ---")
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_cnn_lstm))

print("\nConfusion Matrix (Test Set):")
print(confusion_matrix(y_test, y_pred_cnn_lstm))